# TFM Tenerife — Ingesta de la Red de Transporte (GTFS TITSA y Metropolitano)

Este notebook descarga los ficheros GTFS de TITSA (guaguas) y de Metropolitano de Tenerife (tranvía), construye las paradas y las líneas de cada red, y las sube a Neon siguiendo el mismo patrón `raw_data` / `processed_data` de los notebooks anteriores.



In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/TFM')

print(f"¡Directorio de trabajo fijado en: {os.getcwd()}")

Mounted at /content/drive
¡Directorio de trabajo fijado en: /content/drive/MyDrive/TFM


## Paso 1 — Instalar dependencias

In [2]:
!pip install -q geopandas sqlalchemy psycopg2-binary geoalchemy2 shapely requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 6.9 MB/s eta 0:00:00


## Paso 2 — Conectar con Neon



In [10]:
import os
from google.colab import userdata
from sqlalchemy import create_engine, text

os.environ['NEON_CONN'] = userdata.get('NEON_CONN')
engine = create_engine(os.environ['NEON_CONN'], pool_pre_ping=True, pool_recycle=280)

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. Version de PostGIS:', version)

Conectado. Version de PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## Paso 3 — Qué es un GTFS y de dónde lo sacamos

Un GTFS no es un único fichero: es un `.zip` que contiene varias tablas en texto plano (formato CSV). Las que nos interesan para la topología espacial son:

- `stops.txt` — las paradas, con su latitud y longitud (puntos).
- `shapes.txt` — el trazado real de cada línea, punto a punto (líneas).
- `routes.txt` y `trips.txt` — para poder ponerle nombre a cada línea (ej. 'Línea 015').

Ambas fuentes están verificadas, son oficiales y se actualizan a diario:

In [11]:
FUENTES_GTFS = {
    'titsa': {
        'url': 'https://datos.tenerife.es/ckan/dataset/36c2e26f-0d18-4b5a-b214-1636168e0765/resource/9f291323-8b78-453a-9008-4f0e3bfb3ce3/download/fichero-zip-de-google-transit.zip',
        'operador': 'TITSA',
        'modo': 'guagua',
    },
    'metropolitano': {
        'url': 'https://datos.tenerife.es/ckan/dataset/4b83e018-37d9-40a6-b6d1-1df2b91c8117/resource/7b7bbd1f-f53a-4413-a6b4-ea411e18c66d/download/fichero-zip-de-google-transit.zip',
        'operador': 'Metropolitano de Tenerife',
        'modo': 'tranvia',
    },
}

## Paso 4 — Funciones para descargar y leer el GTFS

Un GTFS se descarga como un `.zip` en memoria y se lee directamente de ahí, sin necesidad de guardarlo en disco.

In [12]:
import io
import zipfile
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString


def descargar_gtfs(url):
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    return zipfile.ZipFile(io.BytesIO(resp.content))


def leer_tabla(zf, nombre_fichero):
    with zf.open(nombre_fichero) as f:
        return pd.read_csv(f)

## Paso 5 — Construir las paradas (puntos)

Cada fila de `stops.txt` se convierte en un punto (`Point`) usando su longitud y latitud.

In [13]:
def construir_paradas(zf, operador, modo):
    stops = leer_tabla(zf, 'stops.txt')
    stops = stops.dropna(subset=['stop_lat', 'stop_lon'])
    geometry = [Point(xy) for xy in zip(stops['stop_lon'], stops['stop_lat'])]
    gdf = gpd.GeoDataFrame(stops, geometry=geometry, crs='EPSG:4326')
    gdf['operador'] = operador
    gdf['modo'] = modo
    columnas = ['stop_id', 'stop_name', 'operador', 'modo', 'geometry']
    return gdf[[c for c in columnas if c in gdf.columns]]

## Paso 6 — Construir las líneas de cada ruta

Cada `shape_id` de `shapes.txt` es una secuencia de puntos que, unidos en orden (`shape_pt_sequence`), forman una línea (`LineString`). Le añadimos el nombre de la ruta cruzando con `trips.txt` y `routes.txt`.

In [14]:
def construir_rutas(zf, operador, modo):
    archivos = zf.namelist()
    columnas = ['shape_id', 'route_short_name', 'route_long_name', 'operador', 'modo', 'geometry']

    if 'shapes.txt' not in archivos:
        print('  aviso:', operador, 'no tiene shapes.txt, no se construyen lineas de ruta')
        return gpd.GeoDataFrame(columns=columnas, geometry='geometry', crs='EPSG:4326')

    shapes = leer_tabla(zf, 'shapes.txt')
    trips = leer_tabla(zf, 'trips.txt')
    routes = leer_tabla(zf, 'routes.txt')

    trips_unicos = trips.drop_duplicates(subset='shape_id')[['shape_id', 'route_id']]
    info_rutas = trips_unicos.merge(routes, on='route_id', how='left')

    shapes_sorted = shapes.sort_values(['shape_id', 'shape_pt_sequence'])
    lineas = []
    for shape_id, grupo in shapes_sorted.groupby('shape_id'):
        linea = LineString(zip(grupo['shape_pt_lon'], grupo['shape_pt_lat']))
        lineas.append({'shape_id': shape_id, 'geometry': linea})

    gdf = gpd.GeoDataFrame(lineas, crs='EPSG:4326')
    gdf = gdf.merge(info_rutas, on='shape_id', how='left')
    gdf['operador'] = operador
    gdf['modo'] = modo
    return gdf[[c for c in columnas if c in gdf.columns]]

## Paso 7 — Función para subir una capa a Neon

Subir tal cual a `raw_data` (4326), reproyectar y subir a `processed_data` (32628), crear los índices GIST.

In [15]:
def subir_capa(gdf, nombre):
    gdf.to_postgis(nombre, engine, schema='raw_data', if_exists='replace', index=False)
    gdf_proc = gdf.to_crs(epsg=32628)
    gdf_proc.to_postgis(nombre, engine, schema='processed_data', if_exists='replace', index=False)
    with engine.begin() as conn:
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_raw_' + nombre +
            ' ON raw_data.' + nombre + ' USING GIST (geometry)'
        ))
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_processed_' + nombre +
            ' ON processed_data.' + nombre + ' USING GIST (geometry)'
        ))
    print('  ->', nombre, 'cargada:', len(gdf), 'filas')

## Paso 8 — Ejecutar todo

Descarga ambos GTFS, construye paradas y rutas de cada operador, las junta en dos capas únicas (marcadas con la columna `operador`) y las sube. El zip de TITSA pesa unos 22 MB, puede tardar un minuto.

In [16]:
paradas_todas = []
rutas_todas = []

for clave, info in FUENTES_GTFS.items():
    print()
    print('Procesando', info['operador'], '...')
    zf = descargar_gtfs(info['url'])
    paradas_todas.append(construir_paradas(zf, info['operador'], info['modo']))
    rutas_todas.append(construir_rutas(zf, info['operador'], info['modo']))

paradas = gpd.GeoDataFrame(pd.concat(paradas_todas, ignore_index=True), crs='EPSG:4326')
rutas = gpd.GeoDataFrame(pd.concat(rutas_todas, ignore_index=True), crs='EPSG:4326')

print()
print('Total paradas:', len(paradas))
print('Total lineas (shapes):', len(rutas))

print()
print('Subiendo a Neon...')
subir_capa(paradas, 'gtfs_paradas')
subir_capa(rutas, 'gtfs_rutas')


Procesando TITSA ...

Procesando Metropolitano de Tenerife ...

Total paradas: 3893
Total lineas (shapes): 867

Subiendo a Neon...
  -> gtfs_paradas cargada: 3893 filas
  -> gtfs_rutas cargada: 867 filas


## Paso 9 — Verificar

Contamos cuántas paradas y líneas hay por operador, para confirmar que se cargó tanto TITSA como el tranvía.

In [17]:
with engine.connect() as conn:
    resumen_paradas = pd.read_sql(
        'SELECT operador, modo, COUNT(*) FROM raw_data.gtfs_paradas GROUP BY operador, modo;', conn
    )
    resumen_rutas = pd.read_sql(
        'SELECT operador, modo, COUNT(*) FROM raw_data.gtfs_rutas GROUP BY operador, modo;', conn
    )

display(resumen_paradas)
display(resumen_rutas)

,operador,modo,count
0,Metropolitano de Tenerife,tranvia,25
1,TITSA,guagua,3868


,operador,modo,count
0,Metropolitano de Tenerife,tranvia,4
1,TITSA,guagua,863


## Notas

- Las tablas `gtfs_paradas` y `gtfs_rutas` mezclan TITSA y el tranvía en una sola tabla cada una, distinguidos por la columna `operador`. Es una decisión de diseño para tener una única red de transporte público unificada..
- `stop_times.txt` y `calendar.txt` (horarios y días de servicio) no se han cargado aquí porque no son datos espaciales. Los añadiremos más adelante, cuando lleguemos al punto 4.5 del índice (accesibilidad e isocronas), que sí los necesita.
